# Experiment 3: Ablation Studies

**Purpose:** 10 ablation experiments validating each architectural decision (Section 9).

**Dataset:** Pascal VOC, 100 epochs per ablation, quantized baseline.

**Expected time:** ~8-10 hours total on T4.

| # | Ablation | What Changes |
|---|----------|--------------|
| A4 | ReLU6 vs SiLU with INT8 | Activation function |
| A6 | Resolution sweep | 224, 320, 416, 640 |
| A9 | Mosaic on vs off | Augmentation |

In [ ]:
# === Cell 1: Setup ===
!rm -rf /content/tinyYOLO 2>/dev/null
!git clone https://github.com/ShMazumder/tinyYOLO.git /content/tinyYOLO
%cd /content/tinyYOLO
!pip install -e . -q

import torch, subprocess, os, json, glob
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

print(f"GPU: {torch.cuda.get_device_name(0)}")

# Download VOC
!python -c "from ultralytics.data.utils import check_det_dataset; check_det_dataset('VOC.yaml')" 2>/dev/null

def run_training(cmd, label):
    """Run with real-time output streaming."""
    print(f"\n{'='*70}")
    print(f"  {label}")
    print(f"{'='*70}\n", flush=True)
    proc = subprocess.Popen(
        cmd, shell=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        bufsize=1, universal_newlines=True,
        env={**os.environ, 'PYTHONUNBUFFERED': '1'}
    )
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    status = '✅ OK' if proc.returncode == 0 else f'❌ FAILED (code {proc.returncode})'
    print(f"\n  → {label}: {status}\n", flush=True)
    return proc.returncode

In [ ]:
# === A4: Activation — ReLU6 (quantized) vs SiLU (standard) ===
run_training(
    "python scripts/train.py --task det --variant quantized --data voc.yaml "
    "--imgsz 416 --epochs 100 --seed 42 --warmup 3 --name ablation-a4-relu6",
    "A4: ReLU6 (quantized)"
)
run_training(
    "python scripts/train.py --task det --variant standard --data voc.yaml "
    "--imgsz 416 --epochs 100 --seed 42 --warmup 3 --name ablation-a4-silu",
    "A4: SiLU (standard)"
)
print("✅ A4 complete")

In [ ]:
# === A6: Resolution Ablation — 224, 320, 416, 640 ===
for res in [224, 320, 416, 640]:
    run_training(
        f"python scripts/train.py --task det --variant quantized --data voc.yaml "
        f"--imgsz {res} --epochs 100 --seed 42 --warmup 3 --name ablation-a6-{res}",
        f"A6: Resolution {res}×{res}"
    )
print("✅ A6 complete")

In [ ]:
# === A9: Mosaic On vs Off ===
# Mosaic ON = reuse A4 relu6 run. Train mosaic OFF:
os.environ['TINYYOLO_NO_MOSAIC'] = '1'
run_training(
    "python scripts/train.py --task det --variant quantized --data voc.yaml "
    "--imgsz 416 --epochs 100 --seed 42 --warmup 3 --name ablation-a9-no-mosaic",
    "A9: Mosaic OFF"
)
del os.environ['TINYYOLO_NO_MOSAIC']
print("✅ A9 complete")

In [ ]:
# === Collect All Ablation Results ===
print("=" * 80)
print("  Ablation Results Summary")
print("=" * 80)

ablation_data = []
for f in sorted(glob.glob('experiments/results/ablation-*/config.json')):
    with open(f) as fh:
        cfg = json.load(fh)
    fm = cfg.get('final_metrics', {})
    name = Path(f).parent.name
    row = {
        'experiment': name, 'params_M': cfg.get('params_M', 0),
        'mAP50': fm.get('mAP50', 0),
        'mAP50_95': fm.get('mAP50_95', fm.get('mAP50-95', 0)),
        'precision': fm.get('precision', 0), 'recall': fm.get('recall', 0),
    }
    ablation_data.append(row)
    print(f"  {name:35s}: mAP@50={row['mAP50']:.4f}  params={row['params_M']:.3f}M")

with open('experiments/results/ablation_summary.json', 'w') as f:
    json.dump(ablation_data, f, indent=2)
print(f"\nSaved {len(ablation_data)} results to experiments/results/ablation_summary.json")

In [ ]:
# === Plot Ablation Results ===
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# A6: Resolution ablation
ax = axes[0]
res_data = [(d['experiment'].split('-')[-1], d['mAP50']*100)
            for d in ablation_data if 'a6' in d['experiment']]
if res_data:
    res_data.sort(key=lambda x: int(x[0]))
    resolutions, maps = zip(*res_data)
    ax.bar(resolutions, maps, color=['#E3F2FD', '#90CAF9', '#2196F3', '#0D47A1'], edgecolor='#333')
    ax.set_xlabel('Resolution'); ax.set_ylabel('mAP@50 (%)')
    ax.set_title('A6: Resolution Ablation'); ax.grid(axis='y', alpha=0.3)
    for i, (r, m) in enumerate(zip(resolutions, maps)):
        ax.text(i, m + 0.5, f'{m:.1f}%', ha='center', fontsize=10, fontweight='bold')

# A4: Activation comparison
ax = axes[1]
act_data = [(d['experiment'].split('-')[-1], d['mAP50']*100)
            for d in ablation_data if 'a4' in d['experiment']]
if act_data:
    names, maps = zip(*act_data)
    bars = ax.bar(names, maps, color=['#4CAF50', '#FF9800'], edgecolor='#333', width=0.5)
    ax.set_ylabel('mAP@50 (%)')
    ax.set_title('A4: Activation (ReLU6 vs SiLU)'); ax.grid(axis='y', alpha=0.3)
    for bar, m in zip(bars, maps):
        ax.text(bar.get_x() + bar.get_width()/2, m + 0.5, f'{m:.1f}%',
                ha='center', fontsize=10, fontweight='bold')

plt.suptitle('TinyYOLO — Ablation Studies (VOC, 100 epochs)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('experiments/results/ablation_plots.png', dpi=200, bbox_inches='tight')
plt.show()
print("Saved: experiments/results/ablation_plots.png")

In [ ]:
!cd experiments && zip -r /content/ablation_results.zip results/ablation-*
print("📥 Download: /content/ablation_results.zip")